# <font color="steelblue">Subtipos moleculares de cáncer de mama</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

## <font color="steelblue">Objetivos del proyecto</font>

A partir de perfiles de **expresión génica** (microarray Affymetrix), construir, comparar y **desplegar** un clasificador del **subtipo molecular** de cáncer de mama. Este proyecto entrena una competencia que no aparece en los demás: trabajar con **alta dimensionalidad y pocas muestras** (**p ≫ n**: ~151 filas × decenas de miles de genes). Eso obliga a:

* **Reducir la dimensión / seleccionar variables** (PCA, t-SNE/UMAP, LASSO, RFE, Boruta) **sin fuga**: la selección de genes debe ir **dentro** del `Pipeline`/CV. Hacerlo sobre todo el dataset es el **error cardinal** de la genómica (estimaciones ~100 % falsas).
* **Regularizar** (Ridge, LASSO, Elastic Net) por la **colinealidad extrema** entre genes coexpresados.
* **Validar con rigor** con muy pocas muestras (sobreajuste fácil).
* **Interpretar biológicamente:** identificar los **genes discriminantes** (posibles biomarcadores), no solo el *accuracy*.

## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

Los datos proceden del experimento **GSE45827**, depositado en el repositorio público GEO y recopilado en **CuMiDa** (*Curated Microarray Database*; Feltes et al., 2019), una base de datos de microarrays de cáncer sometidos a un control de calidad y a un preprocesado homogéneo. Contiene **~151 muestras** biológicas de tejido mamario, cada una descrita por un **perfil completo de expresión génica** medido con el microarray **Affymetrix HG-U133 Plus 2.0**, que interroga simultáneamente unas **54.675 sondas**.

Conviene entender qué es un valor de este conjunto. Cada celda de la matriz no es una medida clínica, sino la **intensidad de fluorescencia normalizada** con que una sonda concreta ha hibridado con el ARN de la muestra: un valor **continuo en escala logarítmica** que actúa como estimación de cuánto se está **expresando** ese gen. No hay valores faltantes, y todas las columnas comparten unidad y escala, lo que hace innecesaria la estandarización habitual.

**Estructura del fichero:**

| Elemento | Descripción |
|---|---|
| **Filas** | Una **muestra biológica** (un paciente o una línea celular). Son ~151. |
| **Columnas de sondas** | Una **sonda** (*probe set*) por columna, identificada con la nomenclatura de Affymetrix: un número, un sufijo de calidad y la terminación `_at` (p. ej. `1007_s_at`, `202763_at`). El sufijo `_s_` indica que la sonda reconoce transcritos de varios genes de una misma familia; `_x_` advierte de hibridaciones cruzadas potencialmente inespecíficas. Varias sondas pueden medir **el mismo gen**. |
| `samples` | **Identificador** de la muestra. Debe **eliminarse**: no aporta información y, si el fichero estuviera ordenado por clase, introduciría fuga. |
| `type` | La **variable objetivo**. |

### <font color="steelblue">La variable objetivo `type`</font>

Las seis clases no son homogéneas en naturaleza: cuatro son **subtipos clínicos de tumor primario**, una es **tejido sano** y otra corresponde a **cultivos in vitro**.

| Clase | Naturaleza | Descripción |
|---|---|---|
| `luminal_A` | Subtipo tumoral | Tumor con receptores hormonales positivos (ER+/PR+), sin sobreexpresión de HER2 y con **baja proliferación**. Es el subtipo más frecuente y el de **mejor pronóstico**. |
| `luminal_B` | Subtipo tumoral | También con receptores hormonales positivos, pero con **mayor índice de proliferación** que el luminal A y peor pronóstico. Su distinción respecto del luminal A es **gradual**, no categórica. |
| `HER2` | Subtipo tumoral | Tumor caracterizado por la **sobreexpresión del receptor HER2**, típicamente con receptores hormonales negativos. Agresivo, pero con **terapia dirigida** disponible (trastuzumab). |
| `TNBC` | Subtipo tumoral | **Triple negativo**: sin receptores de estrógeno ni de progesterona y sin sobreexpresión de HER2. Al carecer de dianas terapéuticas, es el de **peor pronóstico**. |
| `normal` | Tejido sano | **Tejido mamario no tumoral**, utilizado como referencia. Solo ~11 muestras. |
| `cell_line` | Cultivo in vitro | **Líneas celulares** inmortalizadas cultivadas en laboratorio, no tejido de paciente. Solo ~14 muestras. |

**Distribución muy desequilibrada:** los subtipos tumorales (luminal A/B, HER2, TNBC) tienen ~29–41 muestras cada uno, mientras que **`normal` (~11)** y **`cell_line` (~14)** son **diminutos**.

> **Decisión de formulación:** las **líneas celulares** tienen un perfil distinto al de tumores primarios y suelen **excluirse** cuando el objetivo es clasificar **subtipos clínicos**. Podéis trabajar con las 6 clases o con los **4 subtipos clínicos** (excluyendo `cell_line` y, opcionalmente, `normal`). **Elegid y justificad.**

### <font color="steelblue">Advertencias metodológicas</font>

1. **El problema $p \gg n$: la advertencia dominante.** Hay unas **54.675 variables para 151 muestras**, es decir, unas **360 variables por cada observación**. En esta situación los datos son **trivialmente separables**: siempre existe algún hiperplano que clasifica perfectamente el entrenamiento, aunque las sondas no guarden ninguna relación con la enfermedad. Cualquier exactitud elevada debe, por tanto, tomarse con extrema cautela. La reducción de dimensión (PCA, filtrado por varianza) o la selección de variables no son opcionales: son parte del planteamiento.

2. **La selección de variables debe hacerse *dentro* de la validación cruzada.** Este es el error más frecuente y más grave en genómica. Si se seleccionan las sondas más discriminantes usando **todo** el conjunto y después se valida por CV, la selección ya ha «visto» las etiquetas de los datos de validación: el resultado será optimista y **falso**. La selección debe encapsularse en un `Pipeline` que se reajuste en cada *fold*.

3. **`cell_line` no es un subtipo, es un artefacto experimental.** Una línea celular cultivada carece de estroma, vasos e infiltrado inmunitario; su perfil de expresión difiere del de un tumor primario **por razones técnicas**, no biológicas. Incluirla infla artificialmente el rendimiento —el modelo aprende a distinguir «cultivo» de «tejido», no a subtipar tumores— y **enmascara** la dificultad real del problema. Es exactamente la decisión de formulación que se os pide justificar.

4. **Clases minoritarias y validación cruzada.** Con ~11 muestras de `normal`, una CV de 10 particiones deja **una sola muestra** de esa clase en cada *fold*. Hay que usar **`StratifiedKFold`** con pocas particiones (3 o 5) y, preferiblemente, **repetida**; y reportar **F1 macro** o **exactitud balanceada** en lugar de exactitud global, que estaría dominada por las clases grandes.

5. **Dónde esperar los errores.** Biológicamente, `luminal_A` y `luminal_B` comparten el mismo eje (ambos ER+) y difieren sobre todo en el **grado de proliferación**: la frontera entre ambos es **continua**. Es previsible que la matriz de confusión concentre ahí sus errores, y sería sospechoso que no lo hiciera.

6. **Cuidado con la nomenclatura `TNBC` / *basal-like*.** Son conceptos **relacionados pero no idénticos**: «triple negativo» se define por inmunohistoquímica (ausencia de ER, PR y HER2), mientras que *basal-like* se define por el **perfil de expresión**. La mayoría de los triple negativos son basal-like, pero el triple negativo engloba además otros subtipos moleculares. Comprueba con `df['type'].value_counts()` cómo se llama exactamente esta clase en tu copia, porque CuMiDa la etiqueta en ocasiones como `basal`.

7. **Interpretabilidad: SHAP sobre 54.000 sondas no significa nada.** Antes de explicar el modelo hay que **reducir** a un número manejable de variables. Y aun entonces, una sonda no es un gen: varias sondas miden el mismo gen, así que la importancia se **repartirá** entre ellas —el mismo problema de las variables correlacionadas, llevado al extremo—.

8. **Efectos de lote y validez externa.** Las muestras proceden de un único experimento y de una única plataforma. Los perfiles de expresión son notoriamente sensibles a **efectos de lote** (fecha de procesado, laboratorio, normalización), de modo que un modelo entrenado aquí difícilmente generalizaría a datos de otra plataforma sin una armonización previa.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **La selección de variables / reducción de dimensión va DENTRO del `Pipeline`** (se reajusta en cada pliegue de la CV). Seleccionar genes mirando **todo** el dataset es **fuga** y produce estimaciones falsamente perfectas.
2. **Partición estratificada** (con clases tan pequeñas, la estratificación es crítica); el *test* solo se toca al final. Dado el tamaño, **apóyate en CV repetida y anidada** más que en un único *split*.
3. **Regulariza:** con p ≫ n y colinealidad, prioriza modelos **regularizados** (LogReg L1/L2/Elastic Net), **SVM** y **Random Forest**; modelos como kNN/Naive Bayes suelen sufrir (compáralo).
4. **Equilibrado solo en *train***; cuidado con SMOTE en dimensión enorme (interpola en un espacio de decenas de miles de genes).
5. **Interpreta:** identifica los **genes** más discriminantes; valida si tienen sentido biológico (p. ej. `ESR1` en luminales, `ERBB2`/HER2).
6. **Reproducibilidad y honestidad:** `random_state` fijado; reporta lo que no funcionó y los límites.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install kagglehub imbalanced-learn scikit-learn shap mygene gradio
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Descarga y carga (equivalente a usar %cd $path y leer el CSV)
path = kagglehub.dataset_download("brunogrisci/breast-cancer-gene-expression-cumida")
print("Ruta:", path, "| Archivos:", os.listdir(path))
breast_cancer = pd.read_csv(os.path.join(path, "Breast_GSE45827.csv"))
breast_cancer = breast_cancer.drop("samples", axis=1)   # quitamos el identificador
print(f"Dimensiones: {breast_cancer.shape[0]:,} filas × {breast_cancer.shape[1]:,} columnas")
breast_cancer.iloc[:5, :6]

# <font color="steelblue">Fase 1 — Comprensión y EDA en alta dimensión</font>

Con decenas de miles de columnas **no** se puede hacer un EDA variable a variable: hay que cambiar de herramientas.

**Tareas obligatorias**
1. **Dimensiones y objetivo.** Confirmad `n` filas × `p` columnas y la distribución de `type`. Constatad el **desequilibrio** y lo **diminutas** que son `normal` y `cell_line`.
2. **Formulación.** Decidid (y justificad) si trabajáis con **6 clases** o con los **4 subtipos clínicos** (excluyendo `cell_line`/`normal`).
3. **Visualización por reducción dimensional.** Como no podéis graficar 54 000 genes, proyectad con **PCA** y **t-SNE/UMAP** coloreando por `type`: ¿se separan los subtipos? (reutilizad los cuadernos de *PCA* y *manifold* del curso).
4. **Escala y rango.** Verificad que las intensidades están en escala comparable (log) y que no hay columnas constantes.
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** con p ≫ n, ¿por qué es tan fácil sobreajustar y obtener un *accuracy* engañoso? ¿Qué métricas (macro-F1, recall por clase) y qué validación usaréis?

# <font color="steelblue">Fase 2 — Preprocesado, selección de variables SIN FUGA y partición</font>

Este es el **núcleo metodológico**.

**2A. Objetivo y formulación**
1. `y = type` (aplicad la decisión de la Fase 1; si excluís `cell_line`/`normal`, hacedlo aquí).

**2B. Partición**
2. **Partición estratificada** *train*/*test*. Con clases de ~11–14 muestras, valorad un *test* pequeño y **apoyar la evaluación en CV** (Fase 7).

**2C. Reducción/selección DENTRO del `Pipeline` (clave)**
3. Montad un **`Pipeline`**: `StandardScaler` → **selección o reducción** → modelo. Opciones:
   * **Selección univariante** (`SelectKBest`, `f_classif`) con `k` genes,
   * **PCA** con `n_components`,
   * **LASSO**/`SelectFromModel`, **RFE** (extra).
4. **Nunca** seleccionéis genes usando todo `X` antes de partir/validar: la selección se reajusta **en cada pliegue**.

> **A responder:** explicad por qué seleccionar los "mejores genes" sobre el dataset completo y luego validar produce un resultado **falsamente perfecto**.

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Comparad **≥5 familias**, priorizando las **robustas en p ≫ n**: **Regresión logística regularizada** (L1/L2/Elastic Net), **SVM** (lineal y RBF), **Random Forest**, **HistGradientBoosting**; incluid también **kNN** y **Naive Bayes** para **contrastar** que suelen rendir peor aquí.
2. **Validación cruzada repetida estratificada** (con la **selección dentro del `Pipeline`**) y métrica **`f1_macro`** o exactitud balanceada.
3. **Tabla** comparativa + comentario sobre por qué unas familias funcionan mejor que otras en alta dimensión.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

Hay **clases diminutas** (`normal`, `cell_line`). Comparad, sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'` (la opción **más segura** en alta dimensión).
3. **Sobremuestreo:** SMOTE — **con cautela**: interpola en un espacio de decenas de miles de genes y puede crear muestras poco realistas; aplicadlo **después** de la reducción dimensional dentro del `ImbPipeline` y comentad sus riesgos.

Reportad **macro-F1** y **recall por clase** (en especial las minoritarias) y razonad la mejor opción.

> **Sin fugas:** equilibrado dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

En alta dimensión, **los hiperparámetros clave** son la **intensidad de regularización** y el **tamaño de la representación** (nº de genes seleccionados `k` o nº de componentes de PCA).

**Tareas obligatorias**
1. Optimizad **conjuntamente** regularización + nº de variables/componentes de los 2–3 mejores.
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y `f1_macro`; búsqueda **sobre el `Pipeline`** (`sel__k`, `clf__C`, etc.).
3. **CV anidada (muy recomendada):** con tan pocas muestras, es la única estimación honesta.
4. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual**: con tan **pocas muestras**, el conjunto puede **no** mejorar (o sobreajustar). Decidlo con honestidad.
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, fuga en la selección e interpretabilidad biológica</font>

El *test* se usa una sola vez (y, por su tamaño, **acompañado de la estimación por CV anidada**).

**Tareas obligatorias**
1. **Métricas finales:** **matriz de confusión**, `classification_report`, **macro-F1**, **recall por clase**.
2. **Demostración de la fuga (clave).** Comparad dos formas de seleccionar genes:
   * **MAL:** `SelectKBest` sobre **todo** `X` y luego CV → estimación **inflada** (a menudo ~100 %).
   * **BIEN:** selección **dentro** del `Pipeline`/CV → estimación **honesta**.
   Mostrad la diferencia y explicad por qué ocurre. **Es el aprendizaje central del proyecto.**
3. **Interpretabilidad biológica.** Identificad los **genes/sondas más discriminantes** (coeficientes de LASSO, importancia de RF, o SHAP). (Extra) Mapead las sondas Affymetrix a **símbolos de gen** (`mygene`) y comprobad si aparecen genes esperables (`ESR1`, `PGR` en luminales; `ERBB2` en HER2; marcadores basales en TNBC).
4. **Discusión crítica:** sobreajuste, tamaño muestral, validez externa entre plataformas, valor de los **biomarcadores** hallados.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** (escalado + selección/reducción + modelo) con `joblib`. Así recibe un perfil de expresión **crudo** y devuelve el subtipo.
2. **Función de predicción:** `predecir_subtipo(perfil)` que tome el **vector de expresión** de una muestra (las mismas columnas que `X`) y devuelva el subtipo y las **probabilidades**.
3. **Interfaz:** dado que son decenas de miles de genes, **no** tiene sentido un formulario campo a campo; cread una app **Gradio** que **suba un CSV** con el perfil de una muestra (o use el panel reducido de genes seleccionados) y muestre la predicción. En Colab da un **enlace público**.
4. (Opcional, nota extra) Reducir a un **panel de pocos genes** (los seleccionados) como "firma" desplegable.

> **Aviso (obligatorio):** herramienta **educativa**; el modelo depende de la **plataforma de microarray** y del preprocesado de CuMiDa, y **no** generaliza sin más a otras tecnologías (RNA-seq) ni sustituye el diagnóstico molecular clínico.

# <font color="steelblue">Pistas y errores típicos</font>

* **La fuga en la selección es EL error.** Elegir los "mejores genes" mirando todo el dataset y luego validar da ~100 % falso. La selección va **dentro** del `Pipeline`/CV.
* **Regulariza.** Con p ≫ n y colinealidad, LogReg L1/L2/Elastic Net, SVM y RF baten a kNN/Naive Bayes; demuéstralo.
* **n minúsculo:** confía en **CV repetida/anidada**, no en un único *split* (un test con 2 muestras `normal` no dice nada).
* **SMOTE con cautela:** interpolar en 54 000 dimensiones genera muestras poco realistas; `class_weight` suele ser más seguro.
* **Interpreta:** un panel de pocos genes (`ESR1`, `ERBB2`…) con sentido biológico vale más que un *accuracy* alto opaco.
* **Despliegue:** guarda el **Pipeline entero**; el modelo depende de la plataforma y del preprocesado.

# <font color="steelblue">Referencias</font>

* Feltes, B. C., Chandelier, E. B., Grisci, B. I. & Dorn, M. (2019). *CuMiDa: Curated Microarray Database*. Journal of Computational Biology.
* Gruosso, T. et al. — experimento **GSE45827** (GEO).
* Cuadernos del curso: *Componentes principales y variantes*, *Métodos de manifold (Isomap/LLE)*, *MDS*, *SVM no lineal*, *Random Forest*, *Equilibrando las muestras*.
